In [ ]:
from elasticsearch_dsl import connections, Index, Search
from elasticsearch_dsl.query import Range, Terms, Term, Exists
from elasticsearch_dsl.aggs import Terms as PerTerms, Cardinality, Min, Max, Missing, Percentiles
from json import loads, load
from pandas import DataFrame, NA
from pathlib import Path
from favicon import get as get_favicon
from requests import get
from tqdm.auto import tqdm
from datetime import datetime, UTC

In [ ]:
ELASTIC_HOST = "https://elasticsearch.srv.webis.de"
ELASTIC_PORT = 9200
ELASTIC_USER = "ajjxp"
# As a way to hide the password at least from the notebook, enter a path to a file here, which only contains the password for Elastic.
ELASTIC_PASSWORD_FILE = Path("~/.local/share/passwords/webis-elasticsearch.txt").expanduser()
INDEX = "corpus_mastodon_statuses*"

OUTPUT_PATH = Path("../data")

NUM_EXPLICIT_INSTANCES = 10
NUM_EXPLICIT_BUCKETS = 3

# Limit the Elastic searches to a specific date range. Crawling started on 2023-12-21.
DATE_AFTER = "2023-12-01T00:00:00"
## Ca. "2024-01-30T12:00:00" is the time when a new version of the fediverse data was gahtered.
DATE_BEFORE = "2024-02-22T00:00:00"

In [ ]:
# Connect to Elastic.
with ELASTIC_PASSWORD_FILE.open("r") as f:
    password = f.readline().strip("\n")
elastic = connections.create_connection(
    hosts=ELASTIC_HOST + ":" + str(ELASTIC_PORT),
    basic_auth=(ELASTIC_USER, password),
    timeout=300
)

In [ ]:
# Prepare the date query
date_query = Range(crawled_at={"gte": DATE_AFTER, "lte": DATE_BEFORE, "format" : "date_hour_minute_second"})
date_query.to_dict()

In [ ]:
base_search: Search = Index(INDEX)\
    .search()\
    .filter(date_query)
total_count = base_search.count()
total_count

In [ ]:
def save_favicon(instance: str, quiet: bool = False) -> Path | None:
    favicon_path = OUTPUT_PATH / "favicons" / f"favicon-{instance}.png"
    favicon_path.parent.mkdir(parents=True, exist_ok=True)
    if favicon_path.exists():
        return favicon_path
    domain = instance
    if instance.count(".") > 1:
        domain = ".".join(instance.split(".")[-2:])
    if not quiet:
        print(f"Searching for favicon for {domain}...")
    try:
        favicons = get_favicon(f"https://{domain}", timeout=2)
    except Exception:
        return None
    favicons = sorted(favicons, key=lambda f: f.width, reverse=True)
    favicons = [favicon for favicon in favicons if favicon.format == "png"]
    if len(favicons) == 0:
        return None
    if not quiet:
        print(f"Downloading favicon from {domain}...")
    favicon_response = get(favicons[0].url, stream=True, timeout=3)
    try:
        with favicon_path.open("wb") as image:
            for chunk in favicon_response.iter_content(1024):
                image.write(chunk)
    except Exception:
        favicon_path.unlink()
        return None
    return favicon_path

In [ ]:
source_instances_count_search: Search = base_search.params(size=0)
source_instances_count_search.aggs\
    .metric(
        "instances", 
        Cardinality(field="instance.keyword")
    )
source_instances_count = source_instances_count_search.execute()\
    .aggs.instances.value
source_instances_count

In [ ]:
source_uris_count_search: Search = base_search.params(size=0)
source_uris_count_search.aggs\
    .metric(
        "uris", 
        Cardinality(field="uri.keyword")
    )
source_uris_count = source_uris_count_search.execute()\
    .aggs.uris.value
source_uris_count

In [ ]:
crawled_instances_search: Search = base_search.params(size=0)
crawled_instances_search.aggs\
    .bucket(
        "instances",
        PerTerms(field="crawled_from_instance.keyword", size=source_instances_count)
    )
crawled_instances = {
    bucket.key
    for bucket in crawled_instances_search.execute().aggs.instances.buckets
}
len(crawled_instances)

In [ ]:
source_unique_post_counts_search: Search = base_search.params(size=0)
source_unique_post_counts_search.aggs\
    .bucket(
        "instances",
        PerTerms(field="instance.keyword", size=source_instances_count)
    )\
    .metric(
        "uris", 
        Cardinality(field="uri.keyword")
    )
    
source_unique_post_counts_buckets = source_unique_post_counts_search.execute().aggs.instances.buckets
source_post_counts = {
    bucket.key: bucket.doc_count
    for bucket in source_unique_post_counts_buckets
}
source_post_counts
source_unique_post_counts = {
    bucket.key: bucket.uris.value
    for bucket in source_unique_post_counts_buckets
}
source_unique_post_counts

In [ ]:
top_source_instances: list[str] = [
    instance
    for instance, _ in sorted(
        source_unique_post_counts.items(), 
        key=lambda b: b[1],
        reverse=True,
    )[:NUM_EXPLICIT_INSTANCES]
]
top_source_instances

In [ ]:
for instance in top_source_instances:
    print(instance, source_unique_post_counts[instance], source_post_counts[instance])

In [ ]:
source_post_stats = {}
for instance in tqdm([
    # *top_source_instances[2:3], 
    *top_source_instances, 
    None
]):
    source_post_stats_search: Search = base_search.params(size=0)
    if instance is not None:
        source_post_stats_search = source_post_stats_search\
        .filter(Term(instance__keyword=instance))

    stats = {}
        
    print("Loading stats.")
    source_post_stats_search.aggs\
        .metric(
            "min_timestamp", 
            Min(field="created_at")
        )\
        .metric(
            "max_timestamp", 
            Max(field="created_at")
        )\
        .metric(
            "timestamp_percentiles", 
            Percentiles(field="created_at")
        )
    source_post_result = source_post_stats_search.execute()
    stats["count_total"] = source_post_counts[instance] if instance is not None else total_count
    stats["count_unique"] = source_unique_post_counts[instance] if instance is not None else source_uris_count
    source_post_aggs = source_post_result.aggs
    stats["min_timestamp"] = datetime.fromtimestamp(
        timestamp=source_post_aggs.min_timestamp.value / 1000, 
        tz=UTC) if source_post_aggs.min_timestamp.value is not None else None
    del source_post_aggs["min_timestamp"]
    stats["max_timestamp"] = datetime.fromtimestamp(
        timestamp=source_post_aggs.max_timestamp.value / 1000, 
        tz=UTC) if source_post_aggs.max_timestamp.value is not None else None
    del source_post_aggs["max_timestamp"]
    for k, v in source_post_aggs.timestamp_percentiles.values.to_dict().items():
        if k.endswith("_as_string"):
            continue
        stats[f"timestamp_percentile_{k}"] = datetime.fromtimestamp(timestamp=v / 1000, tz=UTC) if v is not None else None
    del source_post_aggs["timestamp_percentiles"]

    print("Loading sensitive count.")
    source_post_sensitive_stats_search: Search = source_post_stats_search\
        .filter(Term(sensitive=True))
    source_post_sensitive_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_sensitive_unique"] = source_post_sensitive_stats_search\
        .execute().aggs.uris.value

    print("Loading media count.")
    source_post_media_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="media_attachments.id.keyword"))
    source_post_media_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_media_unique"] = source_post_media_stats_search\
        .execute().aggs.uris.value

    print("Loading media description count.")
    source_post_media_description_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="media_attachments.description.keyword"))
    source_post_media_description_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_media_description_unique"] = source_post_media_description_stats_search\
        .execute().aggs.uris.value

    print("Loading account bot count.")
    source_post_account_bot_stats_search: Search = source_post_stats_search\
        .filter(Term(account__bot=True))
    source_post_account_bot_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_bot_unique"] = source_post_account_bot_stats_search\
        .execute().aggs.uris.value

    print("Loading account group count.")
    source_post_account_group_stats_search: Search = source_post_stats_search\
        .filter(Term(account__group=True))
    source_post_account_group_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_group_unique"] = source_post_account_group_stats_search\
        .execute().aggs.uris.value

    print("Loading account locked count.")
    source_post_account_locked_stats_search: Search = source_post_stats_search\
        .filter(Term(account__locked=True))
    source_post_account_locked_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_locked_unique"] = source_post_account_locked_stats_search\
        .execute().aggs.uris.value

    print("Loading account non-discoverable count.")
    source_post_account_non_discoverable_stats_search: Search = source_post_stats_search\
        .filter(~Term(account__discoverable=True))
    source_post_account_non_discoverable_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_non_discoverable_unique"] = source_post_account_non_discoverable_stats_search\
        .execute().aggs.uris.value

    print("Loading account non-indexable count (treat as opt-out).")
    source_post_account_non_indexable_stats_search: Search = source_post_stats_search\
        .filter(Term(account__noindex=True))
    source_post_account_non_indexable_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_non_indexable_unique"] = source_post_account_non_indexable_stats_search\
        .execute().aggs.uris.value

    print("Loading account non-indexable count (treat as opt-in).")
    source_post_account_non_indexable_opt_in_stats_search: Search = source_post_stats_search\
        .filter(~Term(account__noindex=False))
    source_post_account_non_indexable_opt_in_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_non_indexable_opt_in_unique"] = source_post_account_non_indexable_opt_in_stats_search\
        .execute().aggs.uris.value

    print("Loading account note count.")
    source_post_account_note_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="account.note.keyword"))
    source_post_account_note_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_account_note_unique"] = source_post_account_note_stats_search\
        .execute().aggs.uris.value

    print("Loading reply count.")
    source_post_reply_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="in_reply_to_id.keyword"))
    source_post_reply_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_reply_unique"] = source_post_reply_stats_search\
        .execute().aggs.uris.value

    print("Loading reblog count.")
    source_post_reblog_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="reblog.id.keyword"))
    source_post_reblog_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_reblog_unique"] = source_post_reblog_stats_search\
        .execute().aggs.uris.value

    print("Loading spoiler count.")
    source_post_spoiler_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="spoiler_text.keyword"))
    source_post_spoiler_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_spoiler_unique"] = source_post_spoiler_stats_search\
        .execute().aggs.uris.value

    print("Loading tags count.")
    source_post_tags_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="tags.name.keyword"))
    source_post_tags_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_tags_unique"] = source_post_tags_stats_search\
        .execute().aggs.uris.value

    print("Loading card count.")
    source_post_card_stats_search: Search = source_post_stats_search\
        .filter(Exists(field="card.url.keyword"))
    source_post_card_stats_search.aggs\
        .metric(
            "uris", 
            Cardinality(field="uri.keyword")
        )
    stats["count_card_unique"] = source_post_card_stats_search\
        .execute().aggs.uris.value
    
    source_post_stats[instance] = stats
df = DataFrame([
    {
        "instance": k,
        **v,
    }
    for k, v in source_post_stats.items()
])
df

In [ ]:
is_mastodon: dict[str, bool] = {
    "mastodon.social": True,
    "misskey.io": False,
    "mstdn.jp": True,
    "mstdn.social": True,
    "live-theater.net": False,
    "sportsbots.xyz": False,
    "rss-parrot.net": False,
    "fedibird.com": True,
    "mastodon.online": True,
    "misskey-square.net": False,
}

In [ ]:
print(r"\begin{tabular*}{\linewidth}{@{}cl@{\extracolsep{\fill}}cccrrrrrrrrrrrrr@{}}")
print(r"  \toprule")
cols = [
    r"\multicolumn{2}{@{}l}{\textbf{Instance}}",
    r"\iconCrawledExplicitly",
    r"\iconMastodon",
    r"\textbf{Days}",
    r"\textbf{Unique}",
    r"\multicolumn{1}{@{}c}{\iconRedundancy}",
    r"\multicolumn{1}{@{}c}{\iconReply}",
    r"\multicolumn{1}{@{}c}{\iconReblog}",
    r"\multicolumn{1}{@{}c}{\iconMedia}",
    # r"\multicolumn{1}{@{}c}{\iconMediaAltText}",
    r"\multicolumn{1}{@{}c}{\iconTag}",
    r"\multicolumn{1}{@{}c}{\iconCard}",
    r"\multicolumn{1}{@{}c}{\iconSpoiler}",
    r"\multicolumn{1}{@{}c}{\iconSensitive}",
    # r"\multicolumn{1}{@{}c}{\iconAuthorNote}",
    r"\multicolumn{1}{@{}c}{\phantom{0}\iconAuthorBot}",
    r"\multicolumn{1}{@{}c}{\iconAuthorLocked}",
    r"\multicolumn{1}{@{}c}{\iconAuthorIndexable}",
    r"\multicolumn{1}{@{}c@{}}{\phantom{0}\iconAuthorDiscoverable}",
]
print(r"  " + " & ".join(cols) + r" \\")
print(r"  \midrule")
for _, row in df[df["instance"].notna()].iterrows():
    if row["instance"] == None:
        continue
    favicon_path = save_favicon(row["instance"], quiet=True)
    cols = [
        r"\favicon{" + (row['instance'] if favicon_path is not None else "mastodon.social") + r"}",
        r"\instance{" + row['instance'] + r"}",
        r"\yes" if row["instance"] in crawled_instances else r"\no",
        r"---" if row["instance"] not in is_mastodon else r"\yes" if is_mastodon[row["instance"]] else r"\no",
        f"{(row['timestamp_percentile_99.0'] - row['timestamp_percentile_1.0']).days:d}d",
        f"{row['count_unique']:,d}",
        f"{(row['count_total'] / row['count_unique']):,.0f}",
        f"{(row['count_reply_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_reblog_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_media_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        # f"{(row['count_media_description_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_tags_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_card_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_spoiler_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_sensitive_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        # f"{(row['count_account_note_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_account_bot_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(row['count_account_locked_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(1 - row['count_account_non_indexable_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
        f"{(1 - row['count_account_non_discoverable_unique'] / row['count_unique']):,.0%}".replace("%", r"\%"),
    ]
    print(r"  " + " & ".join(cols) + r" \\")
print(r"  \midrule")
row_crawled = df[df["instance"].isna()].iloc[0]
cols = [
    r"\iconCrawled",
    f"{source_instances_count:,d}" + r"~sources",
    f"{(len(crawled_instances) / source_instances_count):,.0%}".replace("%", r"\%"),
    r"---",
    f"{(row_crawled['timestamp_percentile_99.0'] - row_crawled['timestamp_percentile_1.0']).days:d}d",
    r"\kern-0.5em" + f"{row_crawled['count_unique']:,d}",
    f"{(row_crawled['count_total'] / row_crawled['count_unique']):,.0f}",
    f"{(row_crawled['count_reply_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_reblog_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_media_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    # f"{(row_crawled['count_media_description_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_tags_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_card_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_spoiler_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_sensitive_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    # f"{(row_crawled['count_account_note_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_account_bot_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(row_crawled['count_account_locked_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(1 - row_crawled['count_account_non_indexable_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
    f"{(1 - row_crawled['count_account_non_discoverable_unique'] / row_crawled['count_unique']):,.0%}".replace("%", r"\%"),
]
print(r"  " + " & ".join(cols) + r" \\")
print(r"  \bottomrule")
print(r"\end{tabular*}")